# RAG Assignment — Dense Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** ITF *Rules of Tennis* (2026 English edition)  
**LLM:** Gemini, configured in `config.yaml`  
**Comparison:** both pipelines use the same PDF, chunks, dense Top-10 candidates, and answer prompt.

| Baseline | Re-ranking |
|---|---|
| Bi-Encoder → cosine Top-10 → direct Top-3 → Gemini | Same Top-10 → Cross-Encoder scores all 10 → re-sort → Top-3 → Gemini |

This notebook measures retrieval quality and latency, checks document-specific failure cases,
and exports evidence tables to `results/`. Change chunk size, overlap, top-k, device, and model
names in `config.yaml`. To enable Gemini, copy `.env.example` to `.env` and add your API key.

**Fairness note:** query examples are hypotheses, not assumed wins. A failure is counted only
when the measured target-chunk rank and answer context demonstrate it.

## 1. Environment and configuration

Run from the repository root with the environment described in `requirements.txt`.

In [57]:
from pathlib import Path
import json
import os
import random
import re
import statistics
import time
import warnings

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def locate_project(start: Path = Path.cwd()) -> Path:
    """Find config.yaml in the current directory or one of its parents."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.yaml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the project folder containing config.yaml")

ROOT = locate_project()
CFG = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
load_dotenv(ROOT / ".env")
PDF_PATH = ROOT / CFG["data"]["pdf_path"]
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
LLM_MODEL = os.getenv("GEMINI_MODEL") or CFG["llm"]["model"]

print(f"Project: {ROOT}")
print(f"Python: {__import__('sys').version.split()[0]}")
print(f"Corpus: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024:.0f} KiB)")
print(f"Gemini model: {LLM_MODEL} | API key configured: {bool(GEMINI_API_KEY)}")
print(f"Chunk size: {CFG['chunking']['chunk_size_words']} words | Top-k: "
      f"{CFG['retrieval']['top_k_initial']} → {CFG['retrieval']['top_k_final']}")

Project: /home/tomato/Desktop/RAG-ablation
Python: 3.14.4
Corpus: 2026-rules-of-tennis-english.pdf (1742 KiB)
Gemini model: gemini-3.1-flash-lite | API key configured: True
Chunk size: 180 words | Top-k: 10 → 3


## 2. Extract and clean the PDF

Text is extracted page-by-page with `pypdf`. We remove stand-alone page numbers and normalize line wrapping and typographic characters while preserving paragraph and rule boundaries.

In [58]:
from pypdf import PdfReader

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Corpus PDF not found: {PDF_PATH}")
reader = PdfReader(str(PDF_PATH))
raw_pages = [page.extract_text() or "" for page in reader.pages]
print(f"Extracted {len(raw_pages)} pages and {sum(map(len, raw_pages)):,} raw characters.")
print("Sample extraction:", repr(raw_pages[4][:220]))

Extracted 47 pages and 83,838 raw characters.
Sample extraction: ' 2 \n1. THE COURT \nThe court shall be a rectangle, 78 feet (23.77 m) long and, for singles matches, 27 feet \n(8.23 m) wide. For doubles matches, the court shall be 36 feet (10.97 m) wide. \n \nThe court shall be divided acr'


In [59]:
def clean_page(text: str) -> str:
    """Remove page-number artifacts, normalize typography, and join wrapped lines."""
    replacements = {
        "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-", "\u00a0": " ", "\u00ad": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    output_lines = []
    for raw_line in text.splitlines():
        line = re.sub(r"\s+", " ", raw_line).strip()
        if not line:
            output_lines.append("")
            continue
        if re.fullmatch(r"\d{1,3}", line):  # printed page number, not rule text
            continue
        output_lines.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(output_lines)).strip()

clean_pages = [clean_page(page) for page in raw_pages]
full_text = "\n\n".join(page for page in clean_pages if page)
print(f"Cleaned text: {len(full_text):,} characters, {len(full_text.split()):,} words.")
print(full_text[:650])

Cleaned text: 81,568 characters, 14,398 words.
ITF RULES OF TENNIS
 6

All the rules,
on the go.
Available to
download now.
RULES OF
TENNIS

CONTENTS

FOREWORD 1
Rule 1 THE COURT 2
Rule 2 PERMANENT FIXTURES 3
Rule 3 THE BALL 3
Rule 4 THE RACKET 4
Rule 5 SCORE IN A GAME 5
Rule 6 SCORE IN A SET 6
Rule 7 SCORE IN A MATCH 6
Rule 8 SERVER AND RECEIVER 6
Rule 9 CHOICE OF ENDS AND SERVICE 6
Rule 10 CHANGE OF ENDS 7
Rule 11 BALL IN PLAY 7
Rule 12 BALL TOUCHES A LINE 7
Rule 13 BALL TOUCHES A PERMANENT FIXTURE 7
Rule 14 ORDER OF SERVICE 7
Rule 15 ORDER OF RECEIVING IN DOUBLES 7
Rule 16 THE SERVICE 8
Rule 17 SERVING 8
Rule 18 FOOT FAULT 8
Rule 19 SERVICE FAULT 8
Rule 20 SECOND SERVICE 9
Rule 21 WHE


## 3. Detect sections and create rule-aware chunks

The table of contents labels its entries differently from the body. We detect the **sequential
main rules before the wheelchair-tennis addendum**, so numbered clauses in appendices are not
mistaken for tennis rules. Each `Case N:` and `Note:` stays together as a retrieval unit; long
blocks are split at sentence boundaries with overlap.

In [60]:
# Find the main rules in sequence, bounded before the wheelchair-tennis addendum.
wheelchair_match = re.search(r"(?m)^RULES OF WHEELCHAIR TENNIS$", full_text)
main_end = wheelchair_match.start() if wheelchair_match else len(full_text)
rule_pattern = re.compile(r"(?m)^(\d{1,2})\.\s+([A-Z][A-Z ,\-'()/&]+)$")
main_rules = []
for match in rule_pattern.finditer(full_text[:main_end]):
    number = int(match.group(1))
    if number == len(main_rules) + 1:
        main_rules.append((match.start(), f"Rule {number} — {match.group(2).title()}"))
assert len(main_rules) == 31, f"Expected 31 rules, detected {len(main_rules)}"

section_starts = list(main_rules)
# Addenda and meaningful subheadings inside appendices.
named_headings = {
    "RULES OF WHEELCHAIR TENNIS": "Rules of Wheelchair Tennis",
    "AMENDMENT TO THE RULES OF TENNIS": "Amendment to the Rules of Tennis",
    "CLASSIFICATION OF COURT PACE": "Classification of Court Pace",
    "BALL MARK INSPECTION PROCEDURES": "Ball Mark Inspection Procedures",
    "ELECTRONIC REVIEW PROCEDURES": "Electronic Review Procedures",
}
for heading, label in named_headings.items():
    for match in re.finditer(rf"(?m)^{re.escape(heading)}$", full_text):
        section_starts.append((match.start(), label))

appendix_pattern = re.compile(r"(?m)^APPENDIX\s+(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII)$")
for match in appendix_pattern.finditer(full_text):
    title_lines = full_text[match.end():].splitlines()
    appendix_title = next((line.strip() for line in title_lines if line.strip()), "")
    if not appendix_title.isupper():
        appendix_title = ""
    label = f"Appendix {match.group(1)}" + (f" — {appendix_title.title()}" if appendix_title else "")
    section_starts.append((match.start(), label))

section_starts.sort(key=lambda item: item[0])
# A heading may be detected by only one pattern, but de-duplicate defensively.
section_starts = list({offset: (offset, label) for offset, label in section_starts}.values())
section_starts.sort(key=lambda item: item[0])
sections = []
for i, (start, label) in enumerate(section_starts):
    end = section_starts[i + 1][0] if i + 1 < len(section_starts) else len(full_text)
    body = full_text[start:end].strip()
    body = re.sub(r"^[^\n]*(?:\n|$)", "", body, count=1).strip()
    if label.startswith("Appendix ") and " — " in label:
        title = label.split(" — ", 1)[1].casefold()
        body_lines = body.splitlines()
        for j, line in enumerate(body_lines):
            if line.strip().casefold() == title:
                body_lines.pop(j)
                break
        body = "\n".join(body_lines).strip()
    if len(body.split()) >= 5:
        sections.append({"title": label, "body": body})

print(f"Detected {len(sections)} sections ({sum(s['title'].startswith('Rule ') for s in sections)} numbered rules).")
for section in sections[:8]:
    print(f"  {section['title']}: {len(section['body'].split())} words")

Detected 48 sections (31 numbered rules).
  Rule 1 — The Court: 636 words
  Rule 2 — Permanent Fixtures: 180 words
  Rule 3 — The Ball: 311 words
  Rule 4 — The Racket: 284 words
  Rule 5 — Score In A Game: 273 words
  Rule 6 — Score In A Set: 155 words
  Rule 7 — Score In A Match: 51 words
  Rule 8 — Server And Receiver: 75 words


In [61]:
CHUNK_CFG = CFG["chunking"]
CHUNK_SIZE = int(CHUNK_CFG["chunk_size_words"])
CHUNK_OVERLAP = int(CHUNK_CFG["chunk_overlap_words"])
MIN_CHUNK_WORDS = int(CHUNK_CFG["min_chunk_words"])

CASE_BOUNDARY = re.compile(r"(?=^(?:Case\s+\d+|Note)\s*:)", flags=re.MULTILINE)
SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+")

def split_long_block(text: str) -> list[str]:
    """Split only oversized blocks; repeat up to CHUNK_OVERLAP words between pieces."""
    sentences = [part.strip() for part in SENTENCE_BOUNDARY.split(text) if part.strip()]
    output, current = [], []
    current_size = 0
    for sentence in sentences:
        sentence_size = len(sentence.split())
        if current and current_size + sentence_size > CHUNK_SIZE:
            output.append(" ".join(current))
            overlap = " ".join(" ".join(current).split()[-CHUNK_OVERLAP:]) if CHUNK_OVERLAP else ""
            current = [overlap] if overlap else []
            current_size = len(overlap.split())
        current.append(sentence)
        current_size += sentence_size
    if current:
        output.append(" ".join(current))
    return output

def label_chunk(section_title: str, body: str) -> str:
    case = re.match(r"^(Case\s+\d+|Note)\s*:", body)
    return f"{section_title} — {case.group(1)}" if case else section_title

chunks = []
for section in sections:
    blocks = [b.strip() for b in CASE_BOUNDARY.split(section["body"]) if b.strip()]
    for block in blocks:
        pieces = [block] if len(block.split()) <= CHUNK_SIZE else split_long_block(block)
        for piece in pieces:
            if len(piece.split()) < 5:
                continue
            chunks.append({
                "id": len(chunks), "section": section["title"],
                "title": label_chunk(section["title"], piece),
                "text": piece, "words": len(piece.split()),
            })

assert len(chunks) >= 40, f"Only {len(chunks)} chunks found; inspect PDF extraction and chunking."
chunk_df = pd.DataFrame(chunks)
print(f"Created {len(chunks)} chunks; mean {chunk_df.words.mean():.1f} words, "
      f"median {chunk_df.words.median():.0f}, max {chunk_df.words.max()}.")
display(chunk_df[["id", "title", "words"]].head(15))

Created 150 chunks; mean 101.9 words, median 96, max 180.


,id,title,words
0,0,Rule 1 — The Court,168
1,1,Rule 1 — The Court,174
2,2,Rule 1 — The Court,169
3,3,Rule 1 — The Court,160
4,4,Rule 1 — The Court,63
5,5,Rule 1 — The Court — Note,22
6,6,Rule 2 — Permanent Fixtures,83
7,7,Rule 2 — Permanent Fixtures — Case 1,97
8,8,Rule 3 — The Ball,178
9,9,Rule 3 — The Ball,96


### Corpus checks

The target passages below are ground-truth anchors for the stress-test queries. The checks
prevent reporting a rank for a phrase that the parser failed to capture.

In [62]:
TARGET_PHRASES = [
    "continue to play another point",
    "bird flying over the court",
    "service let does not cancel a previous fault",
    "First service. The whole point must be replayed",
    "If the ball is soft, not broken, the point shall not be replayed",
    "between the singles sideline and the doubles sideline",
    "one or both feet off the ground",
    "allowed two bounces",
    "three (3) unsuccessful appeals per set",
]
def find_target(phrase: str):
    found = [chunk for chunk in chunks if phrase.casefold() in chunk["text"].casefold()]
    return found[0] if found else None
for phrase in TARGET_PHRASES:
    match = find_target(phrase)
    print(f"{'OK' if match else 'MISSING':<7} {phrase}" + (f" → {match['title']}" if match else ""))
assert all(find_target(phrase) for phrase in TARGET_PHRASES), "One or more source anchors were lost."

OK      continue to play another point → Rule 4 — The Racket — Case 4
OK      bird flying over the court → Rule 26 — Hindrance — Case 3
OK      service let does not cancel a previous fault → Rule 22 — The Let During A Service
OK      First service. The whole point must be replayed → Rule 23 — The Let — Case 1
OK      If the ball is soft, not broken, the point shall not be replayed → Rule 3 — The Ball — Case 1
OK      between the singles sideline and the doubles sideline → Rule 18 — Foot Fault — Case 1
OK      one or both feet off the ground → Rule 18 — Foot Fault — Case 2
OK      allowed two bounces → Rules of Wheelchair Tennis
OK      three (3) unsuccessful appeals per set → Electronic Review Procedures


## 4. Bi-Encoder embeddings and FAISS index

The Bi-Encoder embeds the query and each chunk independently; normalized vectors plus FAISS inner
product produce cosine similarity. The Cross-Encoder instead reads each query–chunk pair jointly,
using token-level cross-attention to produce a relevance score. This is more context-sensitive but
requires a separate forward pass for every pair. We use `IndexFlatIP` (exact dense search) so no
approximate-index setting confounds the rank comparison.

In [63]:
import torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = CFG["models"].get("device") or ("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("config.yaml requests CUDA, but torch.cuda.is_available() is False")

encoder = SentenceTransformer(CFG["models"]["bi_encoder"], device=DEVICE)
# Include the source label in the embedded text to disambiguate similar passages.
corpus_texts = [f"{chunk['title']}\n{chunk['text']}" for chunk in chunks]
start = time.perf_counter()
embeddings = encoder.encode(
    corpus_texts, batch_size=32, normalize_embeddings=True,
    show_progress_bar=True, convert_to_numpy=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
embedding_seconds = time.perf_counter() - start

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"Device: {DEVICE}; {index.ntotal} indexed vectors × {embeddings.shape[1]} dimensions")
print(f"Embedding/index build: {embedding_seconds:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Device: cuda; 150 indexed vectors × 384 dimensions
Embedding/index build: 1.52s


In [64]:
cross_encoder = CrossEncoder(CFG["models"]["cross_encoder"], device=DEVICE)
TOP_K = int(CFG["retrieval"]["top_k_initial"])
FINAL_K = int(CFG["retrieval"]["top_k_final"])
assert TOP_K >= FINAL_K > 0

def dense_candidates(query: str, k: int = TOP_K):
    """Return the same initial dense candidates and cosine scores for both pipelines."""
    query_vector = encoder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    scores, ids = index.search(np.ascontiguousarray(query_vector), min(k, len(chunks)))
    return [(int(chunk_id), float(score)) for chunk_id, score in zip(ids[0], scores[0]) if chunk_id >= 0]

def rerank_candidates(query: str, candidates: list[tuple[int, float]]):
    """Score all dense Top-k (query, chunk) pairs with the Cross-Encoder."""
    pairs = [(query, corpus_texts[chunk_id]) for chunk_id, _ in candidates]
    scores = np.asarray(cross_encoder.predict(pairs)).reshape(-1)
    order = np.argsort(-scores)
    return [(candidates[i][0], float(scores[i])) for i in order]

def retrieve_both(query: str) -> dict:
    candidates = dense_candidates(query)
    reranked = rerank_candidates(query, candidates)
    return {
        "query": query,
        "candidates": candidates,
        "baseline_top": [chunk_id for chunk_id, _ in candidates[:FINAL_K]],
        "reranked": reranked,
        "rerank_top": [chunk_id for chunk_id, _ in reranked[:FINAL_K]],
    }

smoke = retrieve_both("How many court sides are used in doubles?")
assert len(smoke["baseline_top"]) == len(smoke["rerank_top"]) == FINAL_K
assert set(smoke["baseline_top"] + smoke["rerank_top"]).issubset(
    {chunk_id for chunk_id, _ in smoke["candidates"]}
)
print("Baseline Top-3:", [chunks[i]["title"] for i in smoke["baseline_top"]])
print("Re-ranked Top-3:", [chunks[i]["title"] for i in smoke["rerank_top"]])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Baseline Top-3: ['Appendix X — Suggestions On How To Mark Out A Court', 'Rule 1 — The Court', 'Rule 1 — The Court']
Re-ranked Top-3: ['Appendix X — Suggestions On How To Mark Out A Court', 'Rule 1 — The Court', 'Appendix X — Suggestions On How To Mark Out A Court — Note']


## 5. Gemini answer generation

Both arms use this exact prompt template, temperature, and output limit. Without a key, the
notebook still completes the retrieval and evaluation and displays a clearly marked placeholder.

In [65]:
PROMPT_TEMPLATE = """You are a precise assistant answering questions about tennis rules.
Use ONLY the provided excerpts from the ITF Rules of Tennis. Cite the rule/case label.
If the excerpts do not support an answer, say: Not covered by the provided excerpts.

EXCERPTS:
{context}

QUESTION: {question}

ANSWER:"""

def make_prompt(question: str, chunk_ids: list[int]) -> str:
    context = "\n\n".join(
        f"[{chunks[i]['title']}]\n{chunks[i]['text']}" for i in chunk_ids
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)

_gemini = None
if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    _gemini = genai.GenerativeModel(LLM_MODEL)

def generate_answer(question: str, chunk_ids: list[int]):
    """Return answer and API latency in seconds; no API request without a key."""
    if _gemini is None:
        labels = ", ".join(chunks[i]["title"] for i in chunk_ids)
        return f"[Gemini skipped: no API key] Retrieved sources: {labels}", np.nan
    start = time.perf_counter()
    response = _gemini.generate_content(
        make_prompt(question, chunk_ids),
        generation_config={
            "temperature": CFG["llm"]["temperature"],
            "max_output_tokens": CFG["llm"]["max_output_tokens"],
        },
    )
    return response.text.strip(), time.perf_counter() - start

print("Gemini ready:", _gemini is not None)
print("Prompt template uses the same excerpts format for both arms.")

Gemini ready: True
Prompt template uses the same excerpts format for both arms.


## 6. Failure-mode queries and rank shifts

Queries cover keyword traps, explicit exceptions, negation/constraints, and document-specific
rules. For each query, we track the target chunk's dense rank among the initial Top-10 and its
re-ranked position. `None` means the target was not retrieved in the initial Top-10.

In [66]:
QUERIES = [
    {
        "mode": "Keyword trap",
        "query": "My racquet's string bed snapped in a rally; can I still use it for my next point?",
        "target": "continue to play another point",
    },
    {
        "mode": "Keyword trap",
        "query": "A bird flying overhead interrupts the rally; what happens to the point?",
        "target": "bird flying over the court",
    },
    {
        "mode": "Exception / interruption",
        "query": "An outside interruption stops play after a first-serve fault; which service does the player get when the point is replayed?",
        "target": "First service. The whole point must be replayed",
    },
    {
        "mode": "Exception / service let",
        "query": "Does a service let cancel a previously called first-service fault?",
        "target": "service let does not cancel a previous fault",
    },
    {
        "mode": "Exception",
        "query": "A ball is soft but still intact at the end of a rally. Must the point be replayed?",
        "target": "If the ball is soft, not broken, the point shall not be replayed",
    },
    {
        "mode": "Negation / constraint",
        "query": "In singles, may the server position behind the baseline in the doubles alley?",
        "target": "between the singles sideline and the doubles sideline",
    },
    {
        "mode": "Negation / constraint",
        "query": "Does a server commit a foot fault if both feet leave the ground during the serve?",
        "target": "one or both feet off the ground",
    },
    {
        "mode": "Document-specific rule",
        "query": "For a wheelchair player, how many bounces may the ball take before it must be returned?",
        "target": "allowed two bounces",
    },
    {
        "mode": "Document-specific rule",
        "query": "What is the limit for incorrect line-call reviews in one set?",
        "target": "three (3) unsuccessful appeals per set",
    },
]

results = []
answer_rows = []
for item in QUERIES:
    target_chunk = find_target(item["target"])
    assert target_chunk is not None
    result = retrieve_both(item["query"])
    dense_ids = [chunk_id for chunk_id, _ in result["candidates"]]
    reranked_ids = [chunk_id for chunk_id, _ in result["reranked"]]
    dense_rank = dense_ids.index(target_chunk["id"]) + 1 if target_chunk["id"] in dense_ids else None
    rerank_rank = reranked_ids.index(target_chunk["id"]) + 1 if target_chunk["id"] in reranked_ids else None
    baseline_answer, baseline_seconds = generate_answer(item["query"], result["baseline_top"])
    rerank_answer, rerank_seconds = generate_answer(item["query"], result["rerank_top"])
    results.append({
        "mode": item["mode"], "query": item["query"],
        "target_chunk": target_chunk["title"],
        "dense_rank_top10": dense_rank, "cross_encoder_rank": rerank_rank,
        "in_baseline_top3": target_chunk["id"] in result["baseline_top"],
        "in_reranked_top3": target_chunk["id"] in result["rerank_top"],
        "baseline_top3": " | ".join(chunks[i]["title"] for i in result["baseline_top"]),
        "reranked_top3": " | ".join(chunks[i]["title"] for i in result["rerank_top"]),
    })
    answer_rows.append({
        "query": item["query"], "answer_baseline": baseline_answer,
        "answer_reranked": rerank_answer,
        "llm_latency_baseline_s": baseline_seconds,
        "llm_latency_reranked_s": rerank_seconds,
    })

rank_df = pd.DataFrame(results)
answers_df = pd.DataFrame(answer_rows)
display(rank_df[["mode", "query", "target_chunk", "dense_rank_top10", "cross_encoder_rank",
                 "in_baseline_top3", "in_reranked_top3"]])

,mode,query,target_chunk,dense_rank_top10,cross_encoder_rank,in_baseline_top3,in_reranked_top3
0,Keyword trap,My racquet's string bed snapped in a rally; ca...,Rule 4 — The Racket — Case 4,1,1,True,True
1,Keyword trap,A bird flying overhead interrupts the rally; w...,Rule 26 — Hindrance — Case 3,1,1,True,True
2,Exception / interruption,An outside interruption stops play after a fir...,Rule 23 — The Let — Case 1,10,3,False,True
3,Exception / service let,Does a service let cancel a previously called ...,Rule 22 — The Let During A Service,3,1,True,True
4,Exception,A ball is soft but still intact at the end of ...,Rule 3 — The Ball — Case 1,1,1,True,True
5,Negation / constraint,"In singles, may the server position behind the...",Rule 18 — Foot Fault — Case 1,1,1,True,True
6,Negation / constraint,Does a server commit a foot fault if both feet...,Rule 18 — Foot Fault — Case 2,1,2,True,True
7,Document-specific rule,"For a wheelchair player, how many bounces may ...",Rules of Wheelchair Tennis,1,1,True,True
8,Document-specific rule,What is the limit for incorrect line-call revi...,Electronic Review Procedures,4,1,False,True


In [67]:
misses = int((~rank_df["in_baseline_top3"]).sum())
rescues = int(((~rank_df["in_baseline_top3"]) & rank_df["in_reranked_top3"]).sum())
print(f"Target absent from baseline Top-{FINAL_K}: {misses}/{len(rank_df)}")
print(f"Target promoted into re-ranked Top-{FINAL_K}: {rescues}/{len(rank_df)}")
for _, row in rank_df.iterrows():
    before = row["dense_rank_top10"] if pd.notna(row["dense_rank_top10"]) else ">10"
    after = row["cross_encoder_rank"] if pd.notna(row["cross_encoder_rank"]) else ">10"
    recovered = (not row["in_baseline_top3"]) and row["in_reranked_top3"]
    print(f"{'RECOVERED' if recovered else 'not recovered':<14} {before!s:>3} → {after!s:<3}  {row['target_chunk']}")
if rescues < 2:
    print("\nFewer than two queries show a Top-3 recovery in this run. "
          "Inspect the ranked candidates and refine queries before claiming the assignment's two cases.")

Target absent from baseline Top-3: 2/9
Target promoted into re-ranked Top-3: 2/9
not recovered    1 → 1    Rule 4 — The Racket — Case 4
not recovered    1 → 1    Rule 26 — Hindrance — Case 3
RECOVERED       10 → 3    Rule 23 — The Let — Case 1
not recovered    3 → 1    Rule 22 — The Let During A Service
not recovered    1 → 1    Rule 3 — The Ball — Case 1
not recovered    1 → 1    Rule 18 — Foot Fault — Case 1
not recovered    1 → 2    Rule 18 — Foot Fault — Case 2
not recovered    1 → 1    Rules of Wheelchair Tennis
RECOVERED        4 → 1    Electronic Review Procedures


In [68]:
from IPython.display import Markdown, display

# Compare actual paired answers for cases where target presence or rank changed.
changed_queries = set(rank_df.loc[
    (rank_df["dense_rank_top10"] != rank_df["cross_encoder_rank"])
    | (rank_df["in_baseline_top3"] != rank_df["in_reranked_top3"]), "query"
])
for row in answers_df.to_dict("records"):
    if row["query"] in changed_queries:
        display(Markdown(
            f"### {row['query']}\n\n"
            f"**Answer 1 — baseline Top-{FINAL_K}:** {row['answer_baseline']}\n\n"
            f"**Answer 2 — re-ranked Top-{FINAL_K}:** {row['answer_reranked']}"
        ))

### An outside interruption stops play after a first-serve fault; which service does the player get when the point is replayed?

**Answer 1 — baseline Top-3:** Not covered by the provided excerpts.

**Answer 2 — re-ranked Top-3:** The player gets a first service. (Rule 23 — The Let — Case 1)

### Does a service let cancel a previously called first-service fault?

**Answer 1 — baseline Top-3:** No. According to Rule 22, a service let does not cancel a previous fault.

**Answer 2 — re-ranked Top-3:** No. According to Rule 22, "a service let does not cancel a previous fault."

### Does a server commit a foot fault if both feet leave the ground during the serve?

**Answer 1 — baseline Top-3:** No. According to Rule 18 — Foot Fault — Case 2, the server is allowed to have one or both feet off the ground.

**Answer 2 — re-ranked Top-3:** No. According to Rule 18 — Foot Fault — Case 2, the server is allowed to have one or both feet off the ground.

### What is the limit for incorrect line-call reviews in one set?

**Answer 1 — baseline Top-3:** Each player (team) is allowed three (3) unsuccessful appeals per set. [Electronic Review Procedures]

**Answer 2 — re-ranked Top-3:** Each player (team) is allowed three (3) unsuccessful appeals per set. [Electronic Review Procedures, Rule 6]

## 7. Latency profiling

We time the baseline retrieval and the complete re-ranking pipeline separately, with warm-up
runs discarded. GPU work is synchronized around timers when CUDA is active. Gemini network
latency is reported separately because it does not represent local retrieval cost.

In [69]:
PROFILE_CFG = CFG["profiling"]
WARMUPS = int(PROFILE_CFG["warmup_runs"])
TIMED_RUNS = int(PROFILE_CFG["timed_runs"])
profile_query = QUERIES[1]["query"]

def sync_device():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def measure(function):
    for _ in range(WARMUPS):
        function()
        sync_device()
    values = []
    for _ in range(TIMED_RUNS):
        sync_device()
        start = time.perf_counter()
        function()
        sync_device()
        values.append(time.perf_counter() - start)
    return np.asarray(values) * 1000

def baseline_only(query):
    return dense_candidates(query)[:FINAL_K]

def retrieval_plus_reranking(query):
    candidates = dense_candidates(query)
    return rerank_candidates(query, candidates)[:FINAL_K]

candidate_pairs = dense_candidates(profile_query)
ce_pairs = [(profile_query, corpus_texts[i]) for i, _ in candidate_pairs]
measurements = {
    "Baseline: Bi-Encoder + FAISS": measure(lambda: baseline_only(profile_query)),
    f"Cross-Encoder stage ({TOP_K} pairs)": measure(lambda: cross_encoder.predict(ce_pairs)),
    "Re-ranking: retrieval + Cross-Encoder": measure(lambda: retrieval_plus_reranking(profile_query)),
}
latency_df = pd.DataFrame([
    {"pipeline_stage": name, "mean_ms": values.mean(), "median_ms": np.median(values),
     "p95_ms": np.percentile(values, 95), "std_ms": values.std(ddof=1)}
    for name, values in measurements.items()
]).round(2)
display(latency_df)
llm_times = answers_df[["llm_latency_baseline_s", "llm_latency_reranked_s"]].to_numpy(dtype=float)
valid_llm_times = llm_times[np.isfinite(llm_times)]
print(f"Gemini mean round-trip: {valid_llm_times.mean():.2f}s" if valid_llm_times.size
      else "Gemini latency omitted (no API key configured).")

,pipeline_stage,mean_ms,median_ms,p95_ms,std_ms
0,Baseline: Bi-Encoder + FAISS,4.83,4.82,5.57,0.52
1,Cross-Encoder stage (10 pairs),123.69,122.90,129.62,3.62
2,Re-ranking: retrieval + Cross-Encoder,128.18,127.98,130.07,1.36


Gemini mean round-trip: 7.54s


## 8. Scaling: why not Cross-Encode the full collection?

A Cross-Encoder processes each `(query, document)` pair jointly, requiring a model forward pass
for every chunk on every query. Bi-Encoder document vectors can be computed once and searched
cheaply; re-ranking only the small candidate set bounds expensive joint inference. The experiment
below extrapolates measured batched Cross-Encoder throughput and compares it with exact FAISS
search on synthetic vectors. `IndexFlatIP` is O(N·d), not an ANN logarithmic index, but dot products
are far cheaper than transformer inference; approximate indexes can help at larger scales.

In [70]:
# Throughput estimate from scoring a representative batch; the larger values are projections,
# not actual full-corpus Cross-Encoder runs.
SAMPLE_N = min(50, len(chunks))
sample_pairs = [(profile_query, corpus_texts[i]) for i in range(SAMPLE_N)]
batch_times = measure(lambda: cross_encoder.predict(sample_pairs)) / 1000
best_batch_seconds = float(batch_times.min())
ms_per_pair = best_batch_seconds * 1000 / SAMPLE_N

scale_rows = []
for n in PROFILE_CFG["scaling_targets"]:
    scale_rows.append({
        "chunks": int(n),
        "estimated_full_scan_ce_min": ms_per_pair * int(n) / 60_000,
        f"estimated_top{TOP_K}_ce_ms": ms_per_pair * TOP_K,
    })
scale_df = pd.DataFrame(scale_rows)

N_SYNTH = int(max(PROFILE_CFG["scaling_targets"]))
rng = np.random.default_rng(SEED)
synthetic_vectors = rng.standard_normal((N_SYNTH, embeddings.shape[1]), dtype=np.float32)
faiss.normalize_L2(synthetic_vectors)
synthetic_index = faiss.IndexFlatIP(embeddings.shape[1])
synthetic_index.add(synthetic_vectors)
query_vector = np.ascontiguousarray(
    encoder.encode([profile_query], normalize_embeddings=True).astype(np.float32))
faiss_search_ms = float(np.median(measure(
    lambda: synthetic_index.search(query_vector, TOP_K)
)))
scale_df["exact_faiss_100k_search_ms"] = faiss_search_ms
scale_df["estimated_full_ce_vs_faiss_ratio"] = (
    scale_df["estimated_full_scan_ce_min"] * 60_000 / faiss_search_ms)
print(f"Measured CE batch: {SAMPLE_N} pairs in {best_batch_seconds*1000:.1f} ms "
      f"({ms_per_pair:.2f} ms/pair approximate at this batch size)")
display(scale_df.round(2))

Measured CE batch: 50 pairs in 557.4 ms (11.15 ms/pair approximate at this batch size)


,chunks,estimated_full_scan_ce_min,estimated_top10_ce_ms,exact_faiss_100k_search_ms,estimated_full_ce_vs_faiss_ratio
0,1000,0.19,111.47,22.83,488.29
1,10000,1.86,111.47,22.83,4882.90
2,100000,18.58,111.47,22.83,48828.97


## 9. Export and interpretation

CSV outputs preserve the rank shifts, paired answers, latency measurements, and scaling estimates
for use in the concise report. The report itself should be written in your own words. Do not cite
hypothetical recoveries: select at least two cases marked `RECOVERED` above and explain the
observed omission, hallucination, or correction in the paired answers.

In [71]:
rank_df.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
answers_df.to_csv(RESULTS_DIR / "llm_answers.csv", index=False)
latency_df.to_csv(RESULTS_DIR / "latency.csv", index=False)
scale_df.to_csv(RESULTS_DIR / "scaling.csv", index=False)
pd.DataFrame(chunks).to_json(
    RESULTS_DIR / "chunks.json", orient="records", force_ascii=False, indent=2
)
summary = {
    "corpus_pdf": PDF_PATH.name,
    "chunk_count": len(chunks),
    "top_k_initial": TOP_K,
    "top_k_final": FINAL_K,
    "query_count": len(rank_df),
    "baseline_top3_misses": misses,
    "rerank_top3_recoveries": rescues,
    "device": DEVICE,
    "gemini_key_configured": bool(GEMINI_API_KEY),
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved experiment tables to {RESULTS_DIR}")
display(pd.DataFrame([summary]).T)

Saved experiment tables to /home/tomato/Desktop/RAG-ablation/results


,0
corpus_pdf,2026-rules-of-tennis-english.pdf
chunk_count,150
top_k_initial,10
top_k_final,3
query_count,9
baseline_top3_misses,2
rerank_top3_recoveries,2
device,cuda
gemini_key_configured,True
